# 02 · Dataset semantics, grain & arithmetic validation (Phase 2)

**Question:** what does one row represent, and when may rows/values be
aggregated without double-counting or changing meaning?

**Development dataset:** `data/raw/rtt_2026_06.csv` (June 2026) — the single
controlled file. April/May are out of scope.

> **Post-remediation (2026-09-07).** Updated after the independent Codex audit
> (`docs/phase2_independent_audit.md`, PASS WITH CHANGES). Carried through:
> the completed-part unknown-clock column **is** blank on many rows;
> "blank band ≡ zero" is stated only as a **scoped June arithmetic
> convention**, not a semantic conclusion; the **same-month June 2026** SPN is
> the benchmark; NONC pathway counts are per-part with one treatment-function
> representation; the `RTG`/`84H` `Part_2A > Part_2` exception is surfaced.
> See `docs/phase2_remediation_report.md`.

All analysis logic lives in `src/nhs_rtt/semantics.py`; this notebook only
*calls* it. Compact tables → `outputs/phase2/`. Narrative + NHS citations →
`docs/rtt_semantics.md`, `docs/rtt_grain_and_aggregation.md`.

Guard-rails: raw file read-only; week bands **not** reshaped; blanks **not**
imputed in the loaded data; no key manufactured; no KPIs / SQL / Parquet /
modelling.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import pandas as pd
from nhs_rtt import semantics as S

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 220)

RAW = ROOT / "data" / "raw" / "rtt_2026_06.csv"
OUT = ROOT / "outputs" / "phase2"

sha_before = S.assert_raw_unchanged(RAW)
print("extract validation:", S.validate_extract(RAW))
df = S.load_rtt_csv(RAW)              # Int64 numerics; only '' is <NA>; literal NULL kept
band_cols = S.week_band_columns(df.columns.tolist())
print(f"{df.shape[0]:,} rows x {df.shape[1]} cols  |  {len(band_cols)} week-band columns")
print("raw SHA-256:", sha_before)

extract validation: {'path': 'C:\\Users\\abdul\\OneDrive\\Desktop\\P_Projects\\NHS-RTT-Waiting-Times\\data\\raw\\rtt_2026_06.csv', 'n_header_columns': 121, 'duplicate_headers': [], 'n_week_bands': 105, 'expected_band_schema_ok': True, 'problems': [], 'ok': True}


182,411 rows x 121 cols  |  105 week-band columns
raw SHA-256: edc3927e4a0065855ad3b2347e7f82688e687b065406eefb49e2f67a9cd67f02


## Step 1 — Controlled examples

Curated real rows covering every case an analyst must understand: a completed
row with **positive** unknown-clock; a completed row with **blank**
unknown-clock (`Total == Total All`); a normal `Part_2` / `Part_2A` row; the
`RTG`/`84H` `Part_2A > Part_2` data-quality exception; and a `Part_3` row where
`observed_bandsum` is `<NA>` (deliberately **not** shown as 0).

In [2]:
S.controlled_examples(df, band_cols)

,Provider Org Code,Commissioner Org Code,RTT Part Type,Treatment Function Code,Treatment Function Name,observed_bandsum,Total,Patients with unknown clock start date,Total All,note
0,RAJ,06Q,Part_1A,X02,Other - Medical Services,106,106,1,107,"completed, unknown>0: Total + unknown = Total ..."
1,J2H6H,06Q,Part_1A,C_130,Ophthalmology Service,5,5,<NA>,5,"completed, unknown blank: Total = Total All (s..."
2,RAL,93C,Part_2,C_999,Total,108631,<NA>,<NA>,108631,Part_2 snapshot: Total/unknown not collected; ...
3,RAL,93C,Part_2A,C_999,Total,9204,<NA>,<NA>,9204,Part_2A snapshot: Total/unknown not collected;...
4,RTG,84H,Part_2,C_502,Gynaecology Service,1,<NA>,<NA>,1,DATA-QUALITY: RTG/84H/C_502 has Part_2A > Part...
5,RTG,84H,Part_2A,C_502,Gynaecology Service,2,<NA>,<NA>,2,DATA-QUALITY: RTG/84H/C_502 has Part_2A > Part...
6,RTG,84H,Part_2,C_999,Total,1,<NA>,<NA>,1,DATA-QUALITY: RTG/84H/C_999 has Part_2A > Part...
7,RTG,84H,Part_2A,C_999,Total,2,<NA>,<NA>,2,DATA-QUALITY: RTG/84H/C_999 has Part_2A > Part...
8,H3W7Q,06Q,Part_3,C_999,Total,<NA>,<NA>,<NA>,27,Part_3 = count of new clock starts. No band di...


## Step 2/3 — Arithmetic identities, with explicit coverage & missingness

`policy = strict` excludes (and counts) rows with a missing operand — an
unqualified `HOLDS` requires **zero** exclusions over the check's subset.
`policy = missing-as-zero` applies an explicit zero-contribution convention to
a blank operand and records how many rows that touched (`n_missing_operand`).

In [3]:
recon = S.reconciliation_summary(df, band_cols)
recon[["part", "rule", "policy", "subset", "n_in_subset", "n_evaluated",
       "n_missing_operand", "n_excluded_missing", "n_mismatch", "expected", "verdict"]]

,part,rule,policy,subset,n_in_subset,n_evaluated,n_missing_operand,n_excluded_missing,n_mismatch,expected,verdict
0,Part_1A,observed_bandsum == Total,strict,all rows in part,18803,18803,0,0,0,True,HOLDS
1,Part_1A,Total + unknown == Total All,strict,rows where unknown-clock is populated,12771,12771,0,0,0,True,HOLDS
2,Part_1A,Total == Total All,strict,rows where unknown-clock is blank,6032,6032,0,0,0,True,HOLDS
3,Part_1A,Total + unknown(missing->0) == Total All,missing-as-zero,all rows in part,18803,18803,0,0,0,True,HOLDS (missing-as-zero convention)
4,Part_1A,observed_bandsum + unknown(missing->0) == Tota...,missing-as-zero,all rows in part,18803,18803,0,0,0,True,HOLDS (missing-as-zero convention)
5,Part_1B,observed_bandsum == Total,strict,all rows in part,32351,32351,0,0,0,True,HOLDS
6,Part_1B,Total + unknown == Total All,strict,rows where unknown-clock is populated,21346,21346,0,0,0,True,HOLDS
7,Part_1B,Total == Total All,strict,rows where unknown-clock is blank,11005,11005,0,0,0,True,HOLDS
8,Part_1B,Total + unknown(missing->0) == Total All,missing-as-zero,all rows in part,32351,32351,0,0,0,True,HOLDS (missing-as-zero convention)
9,Part_1B,observed_bandsum + unknown(missing->0) == Tota...,missing-as-zero,all rows in part,32351,32351,0,0,0,True,HOLDS (missing-as-zero convention)


In [4]:
exp = recon[recon["expected"]]
assert exp["verdict"].str.startswith("HOLDS").all(), exp[~exp["verdict"].str.startswith("HOLDS")]
assert (exp[exp["policy"] == "strict"]["n_excluded_missing"] == 0).all()
print(f"All {len(exp)} expected checks HOLD; strict checks exclude 0 rows.")

# the two strict subsets partition each completed part:
display(S.reconciliation_coverage_by_part(df))

# strict `observed_bandsum == Total All` 'fails' only where unknown-clock > 0,
# by exactly -unknown:
S.reconciliation_mismatch_examples(df, band_cols, n=6)

All 12 expected checks HOLD; strict checks exclude 0 rows.


,part,n_rows,unknown_present_subset,unknown_blank_subset,subsets_sum,covers_all_rows,note
0,Part_1A,18803,12771,6032,18803,True,strict checks cover both subsets; the two are ...
1,Part_1B,32351,21346,11005,32351,True,strict checks cover both subsets; the two are ...
2,Part_2,63355,0,63355,63355,True,one strict check (observed_bandsum == Total Al...
3,Part_2A,30548,0,30548,30548,True,one strict check (observed_bandsum == Total Al...
4,Part_3,37354,0,37354,37354,True,no distribution identity applies; Total All is...


,Provider Org Code,Commissioner Org Code,RTT Part Type,Treatment Function Code,Treatment Function Name,observed_bandsum,Total,Patients with unknown clock start date,Total All,bandsum_minus_totalall
1648,RAJ,06Q,Part_1A,X02,Other - Medical Services,106,106,1,107,-1
1652,RAJ,06Q,Part_1A,C_999,Total,1052,1052,1,1053,-1
167922,RAL,93C,Part_1A,X05,Other - Surgical Services,183,183,1,184,-1
167924,RAL,93C,Part_1A,C_999,Total,1416,1416,1,1417,-1
124460,RBL,02H,Part_1A,C_502,Gynaecology Service,0,0,1,1,-1
124461,RBL,02H,Part_1A,C_999,Total,0,0,1,1,-1


### Arithmetic model (June; per part, with policy)

| Part | strict identities that HOLD | +convention |
|---|---|---|
| Part_1A / Part_1B | `observed_bandsum = Total` (all rows); `Total + unknown = Total All` (unknown-populated rows: 12,771 / 21,346); `Total = Total All` (unknown-blank rows: 6,032 / 11,005) | `Total + unknown(→0) = Total All` over all rows (convention applied to the blank-unknown rows) |
| Part_2 / Part_2A | `observed_bandsum = Total All` (all rows) | — (`Total`/unknown not collected) |
| Part_3 | none — bands + `Total` + unknown all blank; `observed_bandsum` is `<NA>` | `Total All` is a standalone count of new clock starts |

`observed_bandsum` = the sum of the **observed** band cells. Because every
June banded row has ≥1 observed band, this equals filling blanks with 0 — a
property of June's population, **not** proof that a blank band encodes zero.

## Blank vs explicit zero — arithmetic compatibility, not semantic equivalence

See `docs/rtt_semantics.md` §5. The table describes **represented rows only** —
a missing provider/commissioner/specialty combination cannot be diagnosed from
absence.

In [5]:
S.blank_zero_summary(df, band_cols)

,part,n_rows,Total_blank,Total_zero,Total_pos,unknown_blank,unknown_zero,unknown_pos,TotalAll_blank,TotalAll_pos,band_cells,band_cell_blank,band_cell_zero,band_cell_pos,rows_all_bands_blank,rows_mixing_blank_and_zero_bands
0,Part_1A,18803,0,13,18790,6032,12700,71,0,18803,1974315,248640,1551598,174077,0,551
1,Part_1B,32351,0,46,32305,11005,21111,235,0,32351,3396855,456808,2629830,310217,0,683
2,Part_2,63355,63355,0,0,63355,0,0,0,63355,6652275,879377,5117937,654961,0,1492
3,Part_2A,30548,30548,0,0,30548,0,0,0,30548,3207540,408933,2488409,310198,0,705
4,Part_3,37354,37354,0,0,37354,0,0,0,37354,3922170,3922170,0,0,37354,0


* `Total` / unknown-clock: **100% blank** on Parts 2/2A/3, **never** all-blank
  on 1A/1B → those columns are **not collected** for 2/2A/3.
* Within 1A/1B, unknown-clock is blank on **6,032 / 11,005** rows.
* Part_3: **all 105 bands blank** on every row.
* Band sums reconcile whether or not blank bands are filled with 0, and `0` /
  blank band cells are freely mixed within a row → the **stated June
  convention** "missing band contribution = 0" reproduces the totals. It does
  **not** establish why a cell was serialised blank, nor that the convention
  transfers to another file. NHS Annex B has total/quality checks, not a
  cell-encoding rule.

## Grain & candidate key

Documented grain (S1 §10.1.1.2 / §10.1.4): **Provider × Commissioner ×
Treatment Function × RTT Part** (× Period). `candidate_key_report` returns
`is_unique = None` (never silently `True`) when a column is absent, and
`usable` requires all columns present + no missing key cell + uniqueness.

In [6]:
for k in (
    S.CANDIDATE_KEY,
    [c for c in S.CANDIDATE_KEY if c != "Period"],
    [c for c in S.CANDIDATE_KEY if c != "Commissioner Org Code"],
    [c for c in S.CANDIDATE_KEY if c != "Treatment Function Code"],
):
    r = S.candidate_key_report(df, k)
    print(f"unique={r['is_unique']!s:>5}  usable={r.get('usable')!s:>5}  "
          f"dup_groups={r.get('n_duplicate_groups')}  key={k}")
print("\nmissing-column example:", S.candidate_key_report(df[["Provider Org Code"]], S.CANDIDATE_KEY))

unique= True  usable= True  dup_groups=0  key=['Period', 'Provider Org Code', 'Commissioner Org Code', 'RTT Part Type', 'Treatment Function Code']
unique= True  usable= True  dup_groups=0  key=['Provider Org Code', 'Commissioner Org Code', 'RTT Part Type', 'Treatment Function Code']


unique=False  usable=False  dup_groups=16377  key=['Period', 'Provider Org Code', 'RTT Part Type', 'Treatment Function Code']
unique=False  usable=False  dup_groups=40636  key=['Period', 'Provider Org Code', 'Commissioner Org Code', 'RTT Part Type']

missing-column example: {'key': ['Period', 'Provider Org Code', 'Commissioner Org Code', 'RTT Part Type', 'Treatment Function Code'], 'present_cols': ['Provider Org Code'], 'missing_cols': ['Period', 'Commissioner Org Code', 'RTT Part Type', 'Treatment Function Code'], 'usable': False, 'is_unique': None, 'reason': "requested key column(s) absent: ['Period', 'Commissioner Org Code', 'RTT Part Type', 'Treatment Function Code']", 'n_rows': 182411}


`[Period, Provider Org Code, Commissioner Org Code, RTT Part Type, Treatment
Function Code]` is unique and `usable` for June. Dropping the treatment
function *or* the commissioner breaks uniqueness. It is a **candidate natural
key for this extract**, not an NHS-guaranteed primary key and not a patient/
pathway identifier. Acceptance contract (K-2): all columns present + no missing
key cell + unique **per incoming file** + a revised-release policy. See
`docs/rtt_grain_and_aggregation.md`.

## `C_999` / "Total" and other roll-up structure

`aggregate_vs_detail` compares the `C_999` row against the sum of the
**available** non-C_999 rows, on the **union** of aggregate/detail groups, and
classifies every group×column comparison (numeric/numeric, both-missing,
agg-zero-vs-all-missing-detail).

In [7]:
c999 = S.aggregate_vs_detail(
    df, agg_col="Treatment Function Code", agg_value="C_999",
    group_cols=["Provider Org Code", "Commissioner Org Code", "RTT Part Type"],
    value_cols=band_cols + S.NUMERIC_TAIL_COLS,
)
for k in ("n_common_groups", "n_agg_only_groups", "n_detail_only_groups",
          "n_groups_multiple_agg_rows", "detail_rows_per_group_min",
          "detail_rows_per_group_max", "n_groups_one_detail_row",
          "n_groups_at_max_detail_rows", "cmp_numeric_both", "cmp_numeric_mismatch",
          "cmp_both_missing", "cmp_agg_zero_vs_all_missing_detail",
          "reconciles_numeric"):
    print(f"  {k:<38} {c999[k]}")

S.treatment_function_rows(df)

  n_common_groups                        40636
  n_agg_only_groups                      0
  n_detail_only_groups                   0
  n_groups_multiple_agg_rows             0
  detail_rows_per_group_min              1
  detail_rows_per_group_max              23
  n_groups_one_detail_row                20518
  n_groups_at_max_detail_rows            18
  cmp_numeric_both                       2872115
  cmp_numeric_mismatch                   0
  cmp_both_missing                       928416
  cmp_agg_zero_vs_all_missing_detail     588157
  reconciles_numeric                     True


,Treatment Function Code,Treatment Function Name,n_rows,is_total_rollup_marker
0,C_100,General Surgery Service,9149,False
1,C_101,Urology Service,9047,False
2,C_110,Trauma and Orthopaedic Service,15507,False
3,C_120,Ear Nose and Throat Service,8673,False
4,C_130,Ophthalmology Service,13028,False
5,C_140,Oral Surgery Service,7219,False
6,C_150,Neurosurgical Service,2218,False
7,C_160,Plastic Surgery Service,4279,False
8,C_170,Cardiothoracic Surgery Service,1138,False
9,C_300,General Internal Medicine Service,1596,False


`C_999` = the sum of the group's **available** non-C_999 rows on every numeric
column, **0** numeric mismatches under the stated zero-contribution convention;
40,636 groups each with exactly one C_999 row, ≥1 detail row; **detail rows per
group range 1–23** (only 18 groups carry all 23). 928,416 comparisons are
both-missing and 588,157 are C_999-zero vs all-missing detail — reported, not
counted as observed equality.

**Rule (supported):** for one provider/commissioner/period/part use `C_999`
alone, *or* the reported non-C_999 rows alone — never both.

No *additional* treatment-function roll-up marker was observed beyond `C_999`
("none observed", not a universal guarantee). `Provider Parent` /
`Commissioner Parent` are labels on detail rows — no parent-level aggregate
rows in this extract.

## `Part_2A` ⊆ `Part_2` — documented rule and June conformance

S1 Annex B: `Part_2A` count ≤ `Part_2` count for matching provider /
commissioner / treatment function. Patient membership isn't testable here; the
count implication is.

In [8]:
conf = S.part_2a_subset_conformance(df)
print({k: v for k, v in conf.items() if k != "violations"})
conf["violations"]

{'rule': 'NHS S1 Annex B: Part_2A count <= Part_2 count for matching provider/commissioner/treatment function/period', 'n_part_2a_groups': 30548, 'n_with_matching_part_2': 30548, 'n_without_matching_part_2': 0, 'n_conformant': 30546, 'n_violations': 2, 'disposition': 'Preserve source values; flag violations; do not cap Part_2A or alter Part_2. Any measure relying on Part_2A <= Part_2 must validate this invariant first.'}


,Provider Org Code,Commissioner Org Code,Treatment Function Code,part_2a,part_2,has_matching_part_2
0,RTG,84H,C_502,2,1,True
1,RTG,84H,C_999,2,1,True


**Two June exceptions** (`RTG` / `84H` / `C_502` and its `C_999`): `Part_2A =
2` vs `Part_2 = 1`. A **data-quality exception**, not a refutation of the NHS
rule. Disposition (D-022): preserve source values, flag; never cap `Part_2A`
or alter `Part_2`; any `Part_2A <= Part_2` measure must validate the invariant
per group first. Carried forward as P2-U7.

## `NONC` — prevalence separated from pathway counts

S1 §10.1.8 / §10.1.9.2: `NONC` = non-English commissioner. S5: NONC submission
is **not mandatory** — NONC rows are not a complete record of non-English
activity. **Structural prevalence** (no pathway total — summing across
TFC+parts double-counts) vs **per-part counts using one TFC representation**.

In [9]:
print("prevalence:", S.nonc_prevalence(df))
print("raw checksum (uninterpreted):", S.nonc_raw_cell_checksum(df))
S.nonc_pathways_by_part(df, representation="C_999")

prevalence: {'n_rows': 2993, 'n_providers': 119, 'n_treatment_functions': 24, 'rtt_parts': ['Part_1A', 'Part_1B', 'Part_2', 'Part_2A', 'Part_3'], 'share_of_all_rows_pct': 1.641, 'coverage_note': "NONC submission is not mandatory (S5, 'Navigating published files'); these rows are not a complete record of non-English-commissioned activity."}
raw checksum (uninterpreted): {'raw_total_all_sum_all_nonc_rows': 102482, 'warning': 'double-counts C_999 with its detail rows and mixes RTT Part Types (incompatible/overlapping event bases); not a count of pathways or patients.'}


,RTT Part Type,pathways,representation,do_not_sum_across_parts
0,Part_1A,1224,C_999,True
1,Part_1B,4478,C_999,True
2,Part_2,31156,C_999,True
3,Part_2A,7734,C_999,True
4,Part_3,6649,C_999,True


Per-part NONC pathways (C_999): 1,224 / 4,478 / 31,156 / 7,734 / 6,649 —
**do not sum these five**. The old 102,482 is a raw-cell checksum (C_999 +
detail, mixed parts), not a population. Retain NONC rows; exclude
`Commissioner Org Code = NONC` for England published performance (S5).

## Same-month benchmark — June 2026 SPN Table 1 (unestimated)

S5 fixes the CSV filters; S6 (published 13 Aug 2026) Table 1 gives the
same-month **unestimated** England totals. The estimate-inclusive headline
(S6 page 1) additionally uplifts for non-submitting trusts RHQ, RA9 — not
computed here (P2-U1).

In [10]:
display(S.same_month_totals_check(df))
r = S.incomplete_within_18wk(df, exclude_nonc=True)
print("within-18wk (Part_2 / Total / excl NONC):", r["pct_within_18wk"], "%")
print("benchmark:", r["benchmark_june_2026_spn_unestimated"])

,measure,raw_file_value,june_spn_unestimated,matches
0,incomplete (Part_2),7147562,7147562,True
1,completed admitted (Part_1A),318650,318650,True
2,completed non-admitted (Part_1B),1307837,1307837,True
3,new RTT periods (Part_3),1930912,1930912,True


within-18wk (Part_2 / Total / excl NONC): 65.826 %
benchmark: {'total_all': 7147562, 'pct_within_18wk_1dp': 65.8, 'total_all_matches': True, 'pct_matches_1dp': True, 'source': 'NHS England RTT SPN June 2026, Table 1 (published 13 Aug 2026), not including estimates for missing acute trusts.'}


## Write compact outputs + re-verify raw immutability

In [11]:
outputs = S.run_phase2_analysis(df, source_path=RAW)
for w in outputs.write(OUT):
    print(" ", w.relative_to(ROOT))

assert S.assert_raw_unchanged(RAW, sha_before) == sha_before
print("\nraw file unchanged:", sha_before)

  outputs\phase2\reconciliation_by_part.csv
  outputs\phase2\reconciliation_coverage_by_part.csv
  outputs\phase2\blank_zero_by_part.csv
  outputs\phase2\treatment_function_rows.csv
  outputs\phase2\nonc_pathways_by_part_c999.csv
  outputs\phase2\part2a_subset_violations.csv
  outputs\phase2\controlled_examples.csv
  outputs\phase2\mismatch_positive_unknown.csv
  outputs\phase2\same_month_totals_check.csv
  outputs\phase2\facts.json

raw file unchanged: edc3927e4a0065855ad3b2347e7f82688e687b065406eefb49e2f67a9cd67f02


## Conclusions (evidence-classified)

Full detail + NHS citations: `docs/rtt_semantics.md`,
`docs/rtt_grain_and_aggregation.md`. Remediation status:
`docs/phase2_remediation_report.md`.

| # | Finding | Class |
|---|---|---|
| 1 | Row grain = Provider × Commissioner × Treatment Function × RTT Part (× Period). | CONFIRMED — DOCS + DATA |
| 2 | `[Period, Provider, Commissioner, RTT Part, Treatment Function Code]` is unique & `usable` for June (candidate natural key, not an NHS PK, not a patient id). | CONFIRMED — DATA |
| 3 | 105 week bands are weekly wait bands (by days waited) for Parts 1A/1B/2/2A; Part_3 has none. | CONFIRMED — DOCS + DATA |
| 4 | Completed parts: `observed_bandsum = Total`; `Total All = Total + unknown` on unknown-populated rows and `= Total` on the 6,032 / 11,005 unknown-blank rows (unified over all rows only via the stated missing-as-zero convention). | CONFIRMED — DATA |
| 5 | Incomplete parts: `observed_bandsum = Total All`; `Total`/unknown not collected. Part_3: `Total All` only (count). | CONFIRMED — DATA / DOCS |
| 6 | Blank vs 0: for June the *arithmetic* reconciles treating a missing band contribution as 0 (scoped convention). Semantic meaning of a blank is **unresolved**. Blank `Total`/unknown for 2/2A/3 = not-collected. | ARITHMETIC CONVENTION (DATA) |
| 7 | `C_999` = sum of the group's available non-C_999 rows, 0 numeric mismatches under the convention; combining C_999 with reported detail double-counts. Detail rows/group range 1–23. | DOCS (rule) + DATA |
| 8 | `Part_2A` ⊆ `Part_2` (S1 Annex B) — **two June count exceptions** (`RTG`/`84H`); preserved & flagged (P2-U7). Never sum across RTT Part Type. | DOCUMENTED + DATA (exception) |
| 9 | `NONC` = non-English commissioner; prevalence 2,993 rows / 119 providers; per-part pathways (C_999) 1,224 / 4,478 / 31,156 / 7,734 / 6,649 (do not sum). Submission optional — not exhaustive. | CONFIRMED — DOCS + DATA |
| 10 | Same-month June SPN unestimated totals (7,147,562 / 318,650 / 1,307,837 / 1,930,912; 65.8%) reproduce **exactly** from the raw file. Estimate-inclusive headline UNRESOLVED (P2-U1). | CONFIRMED — DOCS + DATA |
| U | Unresolved: P2-U1 estimate-inclusive headline; P2-U2 unknown-clock incompletes (no longer claimed negligible); P2-U3 cross-month stability; P2-U4 provider org-type; P2-U5 Period parsing; P2-U6 (narrowed) derived band bounds; P2-U7 `RTG`/`84H` exception. | UNRESOLVED |